# Fase 2: Evaluación Comparativa y Análisis Estadístico

## 1. Introducción

Una vez establecidos los hiperparámetros óptimos para cada algoritmo (mitigando el sesgo de configuración), esta segunda fase tiene como objetivo evaluar el desempeño general y la escalabilidad del modelo propuesto. Se someterá al algoritmo de **Optimización por Colonia de Hormigas (ACO)** a una comparación de rendimiento directa frente a la **Búsqueda Tabú (TS)**, operando bajo las mismas restricciones teóricas y computacionales.

Para garantizar la generalización de los resultados y evitar el sobreajuste topológico, la experimentación no se limitará a la red base, sino que se extenderá a tres instancias de grafos con distintas complejidades.

---

## 2. Hipótesis de Comparación Algorítmica (H2)

La experimentación de esta fase está guiada por la siguiente hipótesis formal:

> *El algoritmo ACO, al fundamentarse en una exploración paralela y la construcción estigmérgica de memoria a largo plazo, presentará diferencias estadísticamente significativas a favor en la minimización de la función objetivo frente a la Búsqueda Tabú. Esta superioridad se acentuará en redes de mayor densidad topológica, donde la trayectoria única de la Búsqueda Tabú será más propensa a caer en óptimos locales inducidos por callejones sin salida, a pesar de su memoria de corto plazo.*

### Justificación Teórica

* **Búsqueda Tabú (Línea Base):** Es un algoritmo de trayectoria única. Su mecanismo para escapar de mínimos locales es una memoria de corto plazo (Lista Tabú). En redes densas o asimétricas, forzar movimientos subóptimos puede derivar en un alto consumo energético por *deadheading*, consumiendo la batería antes de lograr una exploración significativa.
* **ACO (Modelo Propuesto):** Emplea una población de $m$ hormigas construyendo soluciones en paralelo. La matriz de feromonas actúa como una memoria colectiva distributiva que, balanceada con el factor heurístico local ($\eta_{ij} = 1 / (c_{ij} \cdot \Omega)$), permite esquivar zonas redundantes (callejones) de manera emergente sin depender de un historial rígido de movimientos prohibidos.

---

## 3. Diseño Experimental y Metodología

Para aislar la variabilidad estocástica y obtener validez estadística, se aplicará el siguiente protocolo riguroso:

1. **Banco de Pruebas Inmutable:** Se utilizarán tres redes serializadas para asegurar que ambos algoritmos enfrenten la misma topología:
   * `red_base_5x5.json`: Grafo estándar de control.
   * `red_densa_10x10.json`: Grafo de alta complejidad combinatoria.
   * `red_realista_asim.json`: Grafo con perturbaciones asimétricas simulando un entorno urbano no planificado.

2. **Control Estocástico:**
   Se ejecutarán **15 corridas independientes** por cada configuración (Algoritmo $\times$ Red). Se utilizará una secuencia de semillas deterministas (1000 a 1014) inyectadas en `numpy.random` para garantizar la reproducibilidad exacta del experimento.

3. **Métrica Principal de Evaluación:**
   $$Z = \frac{B_{max}}{\text{Arcos Únicos Inspeccionados}}$$

In [3]:
import os
import gc
import json
import joblib
import numpy as np
import dataclasses
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from tqdm.notebook import tqdm
from scipy.stats import mannwhitneyu

In [4]:
import sys
from pathlib import Path

src_path = Path().resolve().parent / "src"
sys.path.append(str(src_path))

from aco import ACO_CARP
from tabu import TabuSearch_CARP
from graph_model import RedTuberias

In [5]:
# extracción de parámetros óptimos obtenidos en la Fase 1

estudio_aco = joblib.load("../data/calibracion_aco.pkl")
estudio_tabu = joblib.load("../data/calibracion_tabu.pkl")

In [6]:
PARAM_ACO = {
    **estudio_aco.best_params,
    "n_hormigas": 20,
    "max_iter": 200
}

PARAM_TABU = {
    **estudio_tabu.best_params,
    "max_iter": 500
}

In [7]:
BATERIA_MAX = 15000.0
N_CORRIDAS = 15

RUTA_RESULTADOS = "../data/resultados_fase2.jsonl"

INSTANCIAS = [
    "red_base_5x5.json",
    "red_densa_10x10.json",
    "red_realista_asim.json"
]

In [8]:
# función para guardar resultados con metadatos de la instancia

def guardar_resultado(resultado, instancia_nombre, filepath="./data/resultados_fase2.jsonl"):
    data = dataclasses.asdict(resultado)
    data["parametros"]["instancia_red"] = instancia_nombre

    if data["algoritmo"] == 'ACO':
        data["matriz_feromonas_final"] = data["matriz_feromonas_final"].tolist()  # convertir numpy array a lista para JSON
    
    with open(filepath, 'a', encoding='utf-8') as f:
        f.write(json.dumps(data) + '\n')

---

## 4. Ejecución Computacional

En la siguiente celda se detonará el bucle de experimentación masiva. El proceso generará un total de 90 ejecuciones (3 redes $\times$ 2 algoritmos $\times$ 15 corridas).

In [9]:
# reinicio del dataset de resultados
if os.path.exists(RUTA_RESULTADOS):
    os.remove(RUTA_RESULTADOS)

In [10]:
for archivo_instancia in tqdm(INSTANCIAS, desc="Evaluación de Topologías"):
    ruta_instancia = os.path.join("../data", archivo_instancia)
    
    if not os.path.exists(ruta_instancia):
        print(f"Error estructural: No se halló {archivo_instancia}. Verifique graph_generator.py")
        continue

    for i in tqdm(range(N_CORRIDAS), desc=f"Corridas ({archivo_instancia})", leave=False):
        semilla_control = 1000 + i 
        
        # TABU SEARCH:
        np.random.seed(semilla_control)
        red_tabu = RedTuberias.cargar_red(ruta_instancia)
        ts = TabuSearch_CARP(
            red_tuberias=red_tabu,
            bateria_maxima=BATERIA_MAX,
            **PARAM_TABU
        )
        guardar_resultado(ts.run(), archivo_instancia, RUTA_RESULTADOS)
        del red_tabu, ts
        
        print()

        # ACO:
        np.random.seed(semilla_control)
        red_aco = RedTuberias.cargar_red(ruta_instancia)
        aco = ACO_CARP(
            red=red_aco,
            bateria_max=BATERIA_MAX,
            Q=1.0,
            **PARAM_ACO
        )
        guardar_resultado(aco.run(), archivo_instancia, RUTA_RESULTADOS)
        del red_aco, aco

        gc.collect()    # limpieza de memoria

print("Experimentación concluida. Dataset generado.")

Evaluación de Topologías:   0%|          | 0/3 [00:00<?, ?it/s]

Corridas (red_base_5x5.json):   0%|          | 0/15 [00:00<?, ?it/s]

--- Iniciando Tabu Search ---
Iteraciones: 500 | Tenencia Tabú: 12
Batería Máxima: 15000.0 | Factor de Castigo: 1850.3091219161995

Iteración 1 -> Nuevo Óptimo | Z: 416.67 | Cobertura: 36 tramos | Batería consumida: 14930.00
Iteración 2 -> Nuevo Óptimo | Z: 384.62 | Cobertura: 39 tramos | Batería consumida: 14980.00

Convergencia temprana en iteración 102.

--- Resultados Finales ---
Mejor Puntaje Z (Costo por tramo): 384.62
Tramos únicos inspeccionados: 39 / 40
Batería consumida total: 14980.00
Batería en deadheading: 3100.00
Tiempo de cómputo: 3.7632 seg
Ruta propuesta:
[0, 1, 2, 7, 8, 13, 12, 7, 6, 1, 2, 3, 8, 9, 4, 3, 8, 3, 4, 9, 14, 13, 18, 17, 16, 11, 10, 15, 20, 21, 16, 15, 10, 5, 6, 11, 12, 17, 22, 23, 24, 19, 14, 13, 14, 19, 18, 23, 22, 21, 16, 21, 16]

--- Iniciando ACO ---
Población: 20 hormigas | Iteraciones: 200
Batería Máxima: 15000.0 | Factor de Castigo: 1091.1445640860666

Iteración 1 -> Nuevo Óptimo | Z: 375.00 | Cobertura: 40 tramos | Batería consumida: 14980.00
Deten

Corridas (red_densa_10x10.json):   0%|          | 0/15 [00:00<?, ?it/s]

--- Iniciando Tabu Search ---
Iteraciones: 500 | Tenencia Tabú: 12
Batería Máxima: 15000.0 | Factor de Castigo: 1850.3091219161995

Iteración 1 -> Nuevo Óptimo | Z: 714.29 | Cobertura: 21 tramos | Batería consumida: 14974.00
Iteración 3 -> Nuevo Óptimo | Z: 600.00 | Cobertura: 25 tramos | Batería consumida: 14919.00
Iteración 4 -> Nuevo Óptimo | Z: 576.92 | Cobertura: 26 tramos | Batería consumida: 14986.00
Iteración 7 -> Nuevo Óptimo | Z: 517.24 | Cobertura: 29 tramos | Batería consumida: 14724.00
Iteración 8 -> Nuevo Óptimo | Z: 500.00 | Cobertura: 30 tramos | Batería consumida: 14904.00
Iteración 11 -> Nuevo Óptimo | Z: 483.87 | Cobertura: 31 tramos | Batería consumida: 14970.00
Iteración 16 -> Nuevo Óptimo | Z: 468.75 | Cobertura: 32 tramos | Batería consumida: 14730.00
Iteración 17 -> Nuevo Óptimo | Z: 454.55 | Cobertura: 33 tramos | Batería consumida: 14970.00

Convergencia temprana en iteración 117.

--- Resultados Finales ---
Mejor Puntaje Z (Costo por tramo): 454.55
Tramos úni

Corridas (red_realista_asim.json):   0%|          | 0/15 [00:00<?, ?it/s]

--- Iniciando Tabu Search ---
Iteraciones: 500 | Tenencia Tabú: 12
Batería Máxima: 15000.0 | Factor de Castigo: 1850.3091219161995

Iteración 1 -> Nuevo Óptimo | Z: 483.87 | Cobertura: 31 tramos | Batería consumida: 14974.00
Iteración 2 -> Nuevo Óptimo | Z: 468.75 | Cobertura: 32 tramos | Batería consumida: 14714.00
Iteración 3 -> Nuevo Óptimo | Z: 454.55 | Cobertura: 33 tramos | Batería consumida: 14774.00

Convergencia temprana en iteración 103.

--- Resultados Finales ---
Mejor Puntaje Z (Costo por tramo): 454.55
Tramos únicos inspeccionados: 33 / 45
Batería consumida total: 14774.00
Batería en deadheading: 1520.00
Tiempo de cómputo: 0.8531 seg
Ruta propuesta:
[0, 28, 0, 5, 10, 15, 10, 11, 16, 17, 18, 19, 24, 23, 22, 17, 12, 13, 8, 3, 2, 1, 6, 7, 8, 9, 4, 3, 2, 7, 12, 11, 6, 1, 0, 1, 0, 1, 6, 11, 16, 21, 20, 15]

--- Iniciando ACO ---
Población: 20 hormigas | Iteraciones: 200
Batería Máxima: 15000.0 | Factor de Castigo: 1091.1445640860666

Iteración 1 -> Nuevo Óptimo | Z: 428.57 | C

---

## 5. Recolección de Datos y Primera Visualización

Una vez concluida la simulación, procesamos el archivo JSONL mediante Pandas para tabular los promedios y medianas. Esta estructura tabular servirá de insumo para la redacción del informe final y para las pruebas estadísticas de contraste de hipótesis.

In [11]:
registros = []
with open(RUTA_RESULTADOS, 'r', encoding='utf-8') as f:
    for line in f:
        registros.append(json.loads(line))

In [12]:
df_resultados = pd.json_normalize(registros)

# extraenis las variables de interés
df_analisis = df_resultados[[
    'algoritmo', 
    'parametros.instancia_red', 
    'fitness_final', 
    'arcos_unicos_inspeccionados', 
    'bateria_consumida_total',
    'bateria_consumida_deadheading',
    'tiempo_ejecucion_seg'
]]

In [13]:
# Tabla comparativa de métricas centrales agrupadas por Red y Algoritmo

tabla_resumen = df_analisis.groupby(['parametros.instancia_red', 'algoritmo']).agg(
    Mediana_Fitness=('fitness_final', 'median'),
    Max_Cobertura_Arcos=('arcos_unicos_inspeccionados', 'max'),
    Promedio_Cobertura_Arcos=('arcos_unicos_inspeccionados', 'mean'),
    Bateria_Consumida=('bateria_consumida_total', 'mean'),  
).reset_index()

display(tabla_resumen)

,parametros.instancia_red,algoritmo,Mediana_Fitness,Max_Cobertura_Arcos,Promedio_Cobertura_Arcos,Bateria_Consumida
0,red_base_5x5.json,ACO,375.000000,40,40.000000,14906.666667
1,red_base_5x5.json,Tabu Search,375.000000,40,39.533333,14890.000000
2,red_densa_10x10.json,ACO,468.750000,36,33.000000,14881.866667
3,red_densa_10x10.json,Tabu Search,468.750000,35,31.533333,14923.800000
4,red_realista_asim.json,ACO,416.666667,36,35.666667,14870.666667
5,red_realista_asim.json,Tabu Search,441.176471,36,34.333333,14893.733333


---

## 6. Contraste de Hipótesis (Test U de Mann-Whitney)

Debido a que las metaheurísticas estocásticas no garantizan distribuciones normales en sus resultados (especialmente por la tendencia a agruparse en torno a óptimos locales, lo que genera asimetría), los supuestos de la prueba T de Student se violan.

Por lo tanto, generalmente se aplica el **Test U de Mann-Whitney**, una prueba no paramétrica robusta que compara las medianas de dos muestras independientes.

### Definición de Hipótesis Estadísticas:
* $H_0$ (Hipótesis Nula): No existe diferencia estadísticamente significativa entre las medianas del fitness de ACO y Tabu Search ($\tilde{Z}_{ACO} \ge \tilde{Z}_{TS}$).
* $H_1$ (Hipótesis Alternativa): El fitness mediano de ACO es estrictamente menor al de Tabu Search ($\tilde{Z}_{ACO} < \tilde{Z}_{TS}$).

Se establece un nivel de significancia $\alpha = 0.05$. Un valor $p < \alpha$ justificará el rechazo de la hipótesis nula, aportando validez científica a la dominancia del modelo propuesto.

In [14]:
# agrupamos por cada instancia topológica evaluada

for red in df_analisis['parametros.instancia_red'].unique():
    data_red = df_analisis[df_analisis['parametros.instancia_red'] == red]
    
    # vectores de muestras independientes
    fitness_aco = data_red[data_red['algoritmo'] == 'ACO']['fitness_final']
    fitness_tabu = data_red[data_red['algoritmo'] == 'Tabu Search']['fitness_final']
    
    # queremos probar que ACO es MENOR que Tabu
    stat, p_value = mannwhitneyu(fitness_aco, fitness_tabu, alternative='less')
    
    print(f"Red Evaluada: {red}")
    print(f"  Mediana ACO : {fitness_aco.median():.2f}")
    print(f"  Mediana Tabu: {fitness_tabu.median():.2f}")
    print(f"  Estadístico U: {stat} | p-valor: {p_value:.5e}")
    
    if p_value < 0.05:
        print("  Conclusión: Diferencia significativa a favor de ACO (Rechazo H0).")
    else:
        print("  Conclusión: Sin evidencia para afirmar superioridad estadística.")
    print("-" * 60)

Red Evaluada: red_base_5x5.json
  Mediana ACO : 375.00
  Mediana Tabu: 375.00
  Estadístico U: 60.0 | p-valor: 1.62745e-03
  Conclusión: Diferencia significativa a favor de ACO (Rechazo H0).
------------------------------------------------------------
Red Evaluada: red_densa_10x10.json
  Mediana ACO : 468.75
  Mediana Tabu: 468.75
  Estadístico U: 75.5 | p-valor: 6.06285e-02
  Conclusión: Sin evidencia para afirmar superioridad estadística.
------------------------------------------------------------
Red Evaluada: red_realista_asim.json
  Mediana ACO : 416.67
  Mediana Tabu: 441.18
  Estadístico U: 22.5 | p-valor: 4.62491e-05
  Conclusión: Diferencia significativa a favor de ACO (Rechazo H0).
------------------------------------------------------------


---

## 7. Visualización Empírica (Distribución de Soluciones)

Para complementar la inferencia estadística, generaremos Boxplots. Esta técnica de análisis exploratorio nos permitirá visualizar:
1. La dispersión estocástica intrínseca de cada algoritmo (rango intercuartílico).
2. La vulnerabilidad a óptimos locales (observando la longitud de los "bigotes" y valores atípicos).
3. La degradación algorítmica conforme la densidad topológica de la red aumenta (comparando la base $5\times5$ contra la densa $10\times10$).

In [15]:
# Strip Plot del Fitness (Distribución y Robustez)
fig_strip = px.strip(
    df_analisis,
    x="parametros.instancia_red",
    y="fitness_final",
    color="algoritmo",
    stripmode="group",
    title="Distribución de Resultados y Robustez Algorítmica (Strip Plot)",
    labels={
        "parametros.instancia_red": "Instancia Topológica",
        "fitness_final": "Fitness Final (Z) [Menor es mejor]",
        "algoritmo": "Algoritmo"
    },
    color_discrete_map={"ACO": "#1f77b4", "Tabu Search": "#d62728"}
)

fig_strip.update_traces(jitter=0.2, marker=dict(size=8, opacity=0.7, line=dict(width=1, color='DarkSlateGrey')))
fig_strip.update_layout(template="plotly_white")
fig_strip.show()

In [16]:
# Utilizamos px.box pero activamos el parámetro points="all" para forzar el renderizado de la dispersión junto a la caja.
fig_boxplot = px.box(
    df_analisis,
    x="parametros.instancia_red",
    y="fitness_final",
    color="algoritmo",
    points="all",  # Superposición de observaciones individuales
    title="Distribución y Robustez del Fitness Final (Box-Plot Mixto)",
    labels={
        "parametros.instancia_red": "Instancia Topológica",
        "fitness_final": "Fitness Final (Z) [Menor es mejor]",
        "algoritmo": "Metaheurística"
    },
    color_discrete_map={
        "ACO": "#1f77b4",
        "Tabu Search": "#d62728"
    },
    boxmode="group"
)

fig_boxplot.update_traces(
    jitter=0.2, 
    marker=dict(size=6, opacity=0.7, line=dict(width=1, color='DarkSlateGrey'))
)

fig_boxplot.update_layout(
    template="plotly_white",
    #yaxis=dict(autorange="reversed"),
    legend_title_text="Algoritmo",
    margin=dict(t=50, b=50, l=50, r=50)
)

fig_boxplot.show()

In [17]:
# Utilizamos un Scatter Plot segmentado por red (facet_col) para aislar topologías
fig_scatter = px.scatter(
    df_analisis,
    x="bateria_consumida_deadheading",
    y="arcos_unicos_inspeccionados",
    color="algoritmo",
    facet_col="parametros.instancia_red",
    title="Análisis de Eficiencia: Cobertura vs. Desperdicio por Deadheading",
    labels={
        "bateria_consumida_deadheading": "Batería Desperdiciada (Deadheading)",
        "arcos_unicos_inspeccionados": "Cobertura (Arcos Únicos)",
        "algoritmo": "Metaheurística",
        "parametros.instancia_red": "Red"
    },
    color_discrete_map={
        "ACO": "#1f77b4",
        "Tabu Search": "#d62728"
    }
)

fig_scatter.update_traces(
    marker=dict(size=12, opacity=0.7, line=dict(width=1.5, color='DarkSlateGrey'))
)

fig_scatter.update_layout(
    template="plotly_white",
    legend_title_text="Algoritmo",
    margin=dict(t=50, b=50, l=50, r=50)
)

fig_scatter.update_yaxes(tickformat="d")

fig_scatter.show()

In [29]:
# Derivación de la métrica de energía útil
if 'bateria_inspeccion_util' not in df_resultados.columns:
    df_resultados['bateria_inspeccion_util'] = (
        df_resultados['bateria_consumida_total'] - df_resultados['bateria_consumida_deadheading']
    )

df_resumen_energia = df_resultados.groupby('algoritmo')[
    ['bateria_inspeccion_util', 'bateria_consumida_deadheading']
].mean().reset_index()

fig_desglose = go.Figure()

fig_desglose.add_trace(go.Bar(
    x=df_resumen_energia['algoritmo'],
    y=df_resumen_energia['bateria_inspeccion_util'],
    name='Batería consumida por Inspección',
    marker_color="#68d468",
    text=df_resumen_energia['bateria_inspeccion_util'].round(1),
    textposition='inside',
    textfont=dict(color='black')
))

fig_desglose.add_trace(go.Bar(
    x=df_resumen_energia['algoritmo'],
    y=df_resumen_energia['bateria_consumida_deadheading'],
    name='Batería consumida por Deadheading',
    marker_color="#ff4444",
    text=df_resumen_energia['bateria_consumida_deadheading'].round(1),
    textposition='inside',
    textfont=dict(color='black')
))

fig_desglose.update_traces(width=0.35)

fig_desglose.update_layout(
    title='Consumo Energético Promedio: Inspección vs. Deadheading',
    xaxis_title='Algoritmo',
    yaxis_title='Batería Consumida (Unidades)',
    barmode='stack',
    template='plotly_white',
    height=600,
    width=900,
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="center",
        x=0.5
    ),
    margin=dict(t=80, b=50, l=60, r=50),
    font=dict(family="Arial", size=14)
)

fig_desglose.show()